In [1]:
import os
import re
from dotenv import load_dotenv
from langchain_groq import ChatGroq


from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate, 
    HumanMessagePromptTemplate
)

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\Mateus\AppData\Local\Temp\ipykernel_42968\3873043151.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

## LangSmith for tracking our workflow
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]=os.getenv("LANGCHAIN_TRACING_V2")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGSMITH_ENDPOINT"]=os.getenv("LANGSMITH_ENDPOINT")


print("GROQ:", bool(os.getenv("GROQ_API_KEY")))
print("LANGSMITH:", bool(os.getenv("LANGSMITH_API_KEY")))
print("PROJECT:", os.getenv("LANGCHAIN_PROJECT"))
print("ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))

GROQ: True
LANGSMITH: True
PROJECT: loggin_rag
ENDPOINT: https://api.smith.langchain.com


##### EXTRAÇÃO TEXTO DO DOCUMENTO EM TXT 

In [4]:
document = TextLoader(
    file_path=r"Documentos\Ai_Agents.txt.txt",
    encoding="utf-8"
)

document_loader = document.load()

##### PRE LIMPEZA DO DOCUMENTO CARREGADO 

In [5]:
# 2. Limpeza profunda do texto
for doc in document_loader:
    texto = doc.page_content.replace("\n", " ")
    # Remove sujeiras como  === ou ---, oque atrapala no calculo de similaridade
    texto = re.sub(r'={2,}|-{2,}', '', texto)
    # Substitui  espaços consecutivos por um só
    doc.page_content = re.sub(r'\s+', ' ', texto).strip()

##### SPLITANDO DOCUMENTO EM BLOCOS DE TEXTOS (CHUNKS)

In [6]:
split_document = RecursiveCharacterTextSplitter(
    separators=[" 1.", " 2.", " 3.", " 4.", " 5.", " 6.", " "],
    chunk_size=500,
    chunk_overlap=0
)


chunks = split_document.split_documents(
    document_loader
)



# só pra debugar como estão organizados
for i in range(0,len(chunks)):
    print(f'Chunk numero -> {i}')
    print(chunks[i].page_content)
    print("///////////////")
    print('\n')

Chunk numero -> 0
A REVOLUÇÃO DOS AGENTES DE INTELIGÊNCIA ARTIFICIAL
///////////////


Chunk numero -> 1
1. O Conceito de Agente de IA - Diferente dos modelos de linguagem tradicionais (LLMs) que apenas geram texto, - um Agente de IA é um sistema autônomo projetado para perceber o ambiente, - tomar decisões fundamentadas e executar ações para atingir metas específicas. - O foco principal deixa de ser a simples resposta e passa a ser a execução.
///////////////


Chunk numero -> 2
2. A Arquitetura Fundamental - A espinha dorsal de um agente é composta por quatro pilares essenciais: * Percepção: capacidade de ler dados, textos, imagens e contextos. * Cérebro (LLM): o modelo que raciocina, planeja e decide o próximo passo. * Ferramentas (Tools): integração com APIs, bancos de dados e navegadores. * Memória: armazenamento de curto prazo (contexto) e longo prazo (vetores).
///////////////


Chunk numero -> 3
3. Como Funciona o Ciclo de Ação (ReAct) - O padrão ReAct (Reasoning and Acting) gu

##### CONFIGURANDO O MODELO PARA VETORIZAR CADA CHUNK DO DOCUMENTO

In [7]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

db = Chroma.from_documents(
    chunks,
    embeddings
)

C:\Users\Mateus\AppData\Local\Temp\ipykernel_42968\2764224209.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


##### TRANSFORMANDO A BASE DE DADOS DO CHROMA EM UM RETRIEVER

In [9]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  
)

In [10]:
system_template = (
    "Você é um engenheiro de agentes de IA altamente especializado.\n"
    "Responda à pergunta do usuário utilizando **exclusivamente** o contexto fornecido abaixo.\n"
    "Se a resposta não puder ser encontrada no contexto, responda honestamente que não possui essa informação.\n\n"
    "--- CONTEXTO ---\n"
    "{context}"
)

human_template = "{question}"



prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_template),
    HumanMessagePromptTemplate.from_template(human_template)
])


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


modelo = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.6
)

# 5. Montar a Chain de RAG
rag_chain = (
    {
        "context": retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | prompt
    | modelo
    | StrOutputParser()
)

# 6. Testar o fluxo
resposta = rag_chain.invoke("Quais são o futuro e os desafios ao trabalhar com agentes de ia?")
print(resposta)

Com base no contexto fornecido, o futuro dos agentes de IA está ligado à sua evolução de **suporte passivo** para **autonomia completa nos negócios**. Para que essa transição ocorra de forma eficaz, alguns desafios críticos precisam ser enfrentados:

- **Gerenciamento de alucinações e garantia de saídas confiáveis**  
  - Evitar que o agente produza informações imprecisas ou “alucinações”.  
  - Assegurar que as respostas e ações sejam consistentes e verificáveis.

- **Redução da latência e otimização dos custos de chamadas de API**  
  - Tornar as interações mais rápidas, especialmente em ambientes de produção.  
  - Minimizar o gasto financeiro associado ao uso intensivo de APIs externas.

- **Segurança e alinhamento**  
  - Prevenir a execução de ações indesejadas ou potencialmente prejudiciais.  
  - Garantir que o comportamento do agente esteja alinhado aos objetivos e políticas da organização.

- **Transição do suporte passivo para a autonomia completa**  
  - Evoluir de sistemas